[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C60_Edge_Deployment_Consistency_Course/05_profiling/05_profiling_latency.ipynb)

# 05 · 性能剖析与延迟工程（p99 / roofline / 热降频 / 验收门禁）

目标：把「这个模型跑多快」从一个含糊的口号，变成一组**有测量口径、有统计依据、有验收门限**的数字。

本 notebook 你会亲手实现：
1. **warmup 效应**与「不同步就计时」造出的假 FPS，量化不丢 warmup 会高估多少
2. **p99 需要多少样本**：从二项分布推出 $\sigma_{\text{rank}} = \sqrt{np(1-p)}$，并用蒙特卡洛验证
3. **超时率的算术**：p99 达标 = 每 3.3 秒丢一帧；以及**最长连续超时段**为什么必须单独报
4. **端到端延迟分解**：推理只占 57% → 优化拷贝与预处理后占 91%，端到端省 37%（模型没动）
5. **Roofline**：算子的算术强度、脊点、瓶颈判定，以及 **depthwise 卷积慢于 FLOPs 预测 16.5 倍**
6. **静默回退检查器**：解析 engine 的逐层精度与耗时，五条规则自动亮红灯
7. **批处理的延迟代价**与多相机场景；**热降频模拟**与分段 p99
8. **发布验收门禁**：13 项检查的自动化实现 + 余量趋势

> 心智模型：**数据中心问「平均每秒处理多少」，车端问「最坏情况下这一帧会不会晚」。**

## 1 · 测延迟的七宗罪：warmup 与「不同步」

第一次推理里塞满了 CUDA context 创建、kernel 模块懒加载、handle 初始化、首次显存分配。
下面用一个衰减模型模拟它（首次约 95 ms，时间常数 25 次）。

In [ ]:
import numpy as np

STEADY_MS = 8.20                                  # 稳态 GPU 耗时

def simulate_bench(n, steady=STEADY_MS, first=95.0, tau=25.0, jitter=0.12, seed=0):
    # 模拟一次 benchmark：前若干次因 lazy init / kernel 加载 / 显存分配而极慢
    r = np.random.default_rng(seed)
    i = np.arange(n)
    warm = 1.0 + (first / steady - 1.0) * np.exp(-i / tau)
    return steady * warm + r.gamma(2.0, jitter, n)

lat = simulate_bench(1000)
print(f'第 1 次              : {lat[0]:8.2f} ms   ← 稳态的 {lat[0]/STEADY_MS:.1f} 倍')
print(f'前 10 次均值         : {lat[:10].mean():8.2f} ms')
print(f'前 100 次均值        : {lat[:100].mean():8.2f} ms')
print(f'**全部 1000 次均值** : {lat.mean():8.2f} ms   ← 不丢 warmup 就会报这个数')
print(f'**丢弃前 500 次后**  : {lat[500:].mean():8.2f} ms   ← 这才是稳态延迟')

over = lat.mean() / lat[500:].mean() - 1
print(f'\n不丢 warmup 会**高估 {over:.1%}**')
assert lat[:100].mean() > 2 * STEADY_MS
assert over > 0.15
assert abs(lat[500:].mean() - STEADY_MS - 0.24) < 0.15
print('✅ 规则：至少丢弃前 200-500 次。trtexec 用 --warmUp=500（单位是毫秒，不是次数，注意读文档）。')

In [ ]:
# —— 罪状 2：不同步就计时，会测出荒谬的 FPS ——
ENQUEUE_MS = 0.045          # CPU 把这一次推理提交进 stream 的时间（异步，立刻返回）
GPU_MS = 8.20               # GPU 上真正执行的时间
H2D_D2H_MS = 0.39           # 拷贝
CPU_PRE_POST = 0.70         # CPU 侧预处理尾巴 + 结果整理

fake_fps = 1000.0 / ENQUEUE_MS
gpu_fps = 1000.0 / GPU_MS
e2e_ms = ENQUEUE_MS + GPU_MS + H2D_D2H_MS + CPU_PRE_POST
print(f"{'口径':<34s} {'延迟 ms':>10s} {'FPS':>10s}")
print(f"{'❌ 只测 enqueue（忘了同步）':<34s} {ENQUEUE_MS:>10.3f} {fake_fps:>10.0f}")
print(f"{'GPU Compute Time（CUDA event）':<34s} {GPU_MS:>10.3f} {gpu_fps:>10.1f}")
print(f"{'Host Latency（含拷贝）':<34s} {GPU_MS+H2D_D2H_MS:>10.3f} {1000/(GPU_MS+H2D_D2H_MS):>10.1f}")
print(f"{'✅ 端到端（含预处理与后处理）':<34s} {e2e_ms:>10.3f} {1000/e2e_ms:>10.1f}")

assert fake_fps > 20000, '不同步测出来的 FPS 会是荒谬的量级'
assert e2e_ms > GPU_MS + 1.0, '端到端必然明显大于纯 GPU 时间'
print(f'\n⚠️  {fake_fps:.0f} FPS 与 {1000/e2e_ms:.1f} FPS 相差 {fake_fps/(1000/e2e_ms):.0f} 倍 —— '
      '而两者都能被写成「我们测出来的 FPS」。')
print('✅ 报 FPS 必须声明口径。trtexec 的四个数（GPU Compute / Enqueue / Host Latency / Throughput）')
print('   要分清；Enqueue 接近甚至超过 GPU Compute，说明 CPU 提交跟不上 -> 上 CUDA Graph。')

## 2 · p99 要多少样本才算得准

$n$ 个样本中 $p$ 分位落在第 $\lceil pn \rceil$ 个次序统计量上，
「超过真实 $p$ 分位的样本数」服从 $\mathrm{Bin}(n, 1-p)$，秩的标准差是 $\sigma_{\text{rank}}=\sqrt{np(1-p)}$。

In [ ]:
def rank_sigma(n, p=0.99):
    return float(np.sqrt(n * p * (1 - p)))

print(f"{'n':>8s} {'秩':>8s} {'超过 p99 的样本数':>18s} {'σ_rank':>9s} {'95% 分位区间':>22s}")
for n in [100, 1000, 10000, 100000]:
    s = rank_sigma(n)
    lo, hi = (0.99 * n - 1.96 * s) / n, (0.99 * n + 1.96 * s) / n
    print(f'{n:>8d} {int(np.ceil(0.99*n)):>8d} {int(round(n*0.01)):>18d} {s:>9.2f} '
          f'{"[%.2f%%, %.2f%%]" % (lo*100, hi*100):>22s}')
assert int(round(100 * 0.01)) == 1, 'n=100 时只有 1 个样本超过 p99 —— 你的 p99 就是最大值'
print('\n⚠️  n=100 时「p99」= 最大值，而最大值的方差极大（一次系统中断就能污染它）。')

# —— 蒙特卡洛验证：估计值的离散程度确实按 1/sqrt(n) 收缩 ——
def draw(n, seed):
    return np.random.default_rng(seed).gamma(9.0, 0.9, n) + 4.0

TRUE_P99 = float(np.percentile(draw(2_000_000, 1), 99))
print(f'\n真实 p99（200 万样本）= {TRUE_P99:.4f} ms')
print(f"{'n':>8s} {'估计均值':>10s} {'估计标准差':>12s} {'最大偏差':>10s}")
stds = []
for n in [100, 1000, 10000]:
    est = np.array([np.percentile(draw(n, 5000 + k), 99) for k in range(300)])
    stds.append(est.std())
    print(f'{n:>8d} {est.mean():>10.4f} {est.std():>12.4f} {np.abs(est-TRUE_P99).max():>10.4f}')
assert stds[0] > stds[1] > stds[2], '样本量越大，p99 估计越稳'
assert stds[0] > 3 * stds[2], 'n=100 与 n=10000 的估计离散度应差 3 倍以上'
print(f'\n✅ n=100 的估计标准差是 n=10000 的 {stds[0]/stds[2]:.1f} 倍。')
print('   30 FPS 下 10000 帧 ≈ 5.5 分钟 —— **这是 p99 的最低采样成本**；')
print('   要估 p99.9 则需要约 10 万帧 ≈ 55 分钟（正好也覆盖了热稳态，一举两得）。')

## 3 · 尾延迟与超时账：p99 达标 = 每 3.3 秒丢一帧

合成一条真实形态的延迟序列：**固定部分 + 数据依赖的 NMS（超线性）+ 系统抖动 + 罕见尖峰**。

In [ ]:
FPS, N_FRAME = 30, 200_000

def make_latency(n, seed=0, fixed=6.40):
    r = np.random.default_rng(seed)
    n_obj = np.clip(r.lognormal(np.log(8), 0.9, n).astype(int) + 1, 1, 220)   # 重尾：多数帧目标少
    n_cand = n_obj * 22 + 10 + 3 * n_obj                                       # 候选数随场景增长
    nms = 0.25 + 6e-5 * n_cand ** 1.35                                         # **超线性**（C53 m04）
    sysj = r.gamma(1.5, 0.18, n)                                               # 调度抖动
    spike = (r.random(n) < 0.004) * r.uniform(3.0, 12.0, n)                    # 罕见系统尖峰
    return fixed + nms + sysj + spike, n_obj

lat, n_obj = make_latency(N_FRAME, seed=42)
qs = dict(zip(['p50', 'p90', 'p99', 'p99.9'], np.percentile(lat, [50, 90, 99, 99.9])))
print(f"{'mean':>8s} {'p50':>8s} {'p90':>8s} {'p99':>8s} {'p99.9':>8s} {'max':>8s}")
print(f'{lat.mean():>8.2f} {qs["p50"]:>8.2f} {qs["p90"]:>8.2f} {qs["p99"]:>8.2f} '
      f'{qs["p99.9"]:>8.2f} {lat.max():>8.2f}')
print(f'\n场景目标数分布: p50={np.percentile(n_obj,50):.0f}  p99={np.percentile(n_obj,99):.0f}  '
      f'max={n_obj.max()}')

assert lat.mean() < qs['p99'], '均值对长尾几乎不敏感'
assert qs['p99'] - qs['p50'] > 1.5, 'p99 与 p50 之间应有可观差距'
assert lat.max() > qs['p99.9'] + 1.0, 'max 是单样本，噪声远大于 p99.9'

print(f"\n{'延迟预算':>10s} {'超时率 ε':>10s} {'超时帧/小时':>12s} {'平均多久一次':>14s}")
prev = 1e9
for budget in sorted([8.0, 9.0, qs['p99'], 10.0, 11.0, 12.0]):
    eps = float((lat > budget).mean())
    per_h = eps * FPS * 3600
    gap = (1 / (eps * FPS)) if eps > 0 else float('inf')
    tag = '  ← 恰好 p99' if abs(budget - qs['p99']) < 1e-9 else ''
    print(f'{budget:>10.2f} {eps:>10.2%} {per_h:>12.0f} {gap:>13.1f}s{tag}')
    assert eps <= prev; prev = eps

eps99 = float((lat > qs['p99']).mean())
per_h99 = eps99 * FPS * 3600
assert abs(per_h99 - 1080) < 80, per_h99
print(f'\n⚠️  **「p99 达标」翻译成物理事件 = 每小时 {per_h99:.0f} 帧超时 = 每 {1/(eps99*FPS):.1f} 秒丢一帧。**')
print('   对 30 FPS 的感知链路，这意味着跟踪器每隔几秒就要处理一次「这一帧没有观测」。')
print('   多数量产系统的实际门槛是 p99.9（每 33 秒一次），安全相关任务要到 p99.99。')

In [ ]:
# —— 同一组延迟值，同样的每一个分位数，重排一下顺序，安全性完全不同 ——
def longest_run(mask):
    best = cur = 0
    for v in mask:
        cur = cur + 1 if v else 0
        if cur > best:
            best = cur
    return best

BUDGET = float(qs['p99'])
mask_indep = lat > BUDGET
idx_to = np.where(mask_indep)[0]
idx_ok = np.where(~mask_indep)[0]
# 构造「成簇」版本：把所有超时帧连续地排在一起（模拟热节流 / 突发抢占）
clustered = np.concatenate([lat[idx_ok[:50_000]], lat[idx_to], lat[idx_ok[50_000:]]])
mask_clu = clustered > BUDGET

print(f"{'':<14s} {'mean':>8s} {'p50':>8s} {'p99':>8s} {'p99.9':>8s} {'超时率':>9s} {'最长连续超时':>13s}")
for nm, x, mk in [('独立抖动', lat, mask_indep), ('成簇（热节流）', clustered, mask_clu)]:
    p = np.percentile(x, [50, 99, 99.9])
    print(f'{nm:<14s} {x.mean():>8.2f} {p[0]:>8.2f} {p[1]:>8.2f} {p[2]:>8.2f} '
          f'{mk.mean():>9.2%} {longest_run(mk):>13d}')

assert np.allclose(np.percentile(lat, [50, 90, 99, 99.9]),
                   np.percentile(clustered, [50, 90, 99, 99.9])), '同一组数值，所有分位数完全相同'
assert abs(mask_indep.mean() - mask_clu.mean()) < 1e-12
r_ind, r_clu = longest_run(mask_indep), longest_run(mask_clu)
assert r_clu > 100 * r_ind, (r_ind, r_clu)
print(f'\n⚠️  **每一个统计量都完全相同**（因为是同一组数值的重排），')
print(f'    但「最长连续超时段」从 {r_ind} 帧变成 {r_clu} 帧 = 连续 {r_clu/FPS:.0f} 秒失明。')
print('✅ 所以验收必须同时报 p99 **和** 最长连续超时段 —— 后者是最常被漏掉的指标。')
print('   独立抖动 -> 治调度与分配；成簇 -> 治热节流与多任务抢占（第 7 节）。')

## 4 · 端到端延迟分解：推理往往只占一半

In [ ]:
STAGES = [
    #  阶段            朴素    优化后   优化手段
    ('取帧/解码',       0.30,   0.05,  '相机 dmabuf 直接映射给 CUDA，省一次 memcpy'),
    ('预处理',          3.20,   0.35,  'CPU 单核 -> 融合的 GPU kernel（或 VIC 硬件缩放器）'),
    ('H2D 拷贝',        0.31,   0.08,  '传 uint8 而不是 float32，归一化搬到 GPU 上做'),
    ('推理 FP16',       8.20,   8.20,  '——（本节不碰模型）'),
    ('D2H 拷贝',        0.21,   0.001, 'GPU 侧解码+NMS，只回传 300 个框'),
    ('后处理',          2.10,   0.35,  '解码与 NMS 搬到 GPU（模块 04 方案 B/C）'),
]
base = sum(s[1] for s in STAGES)
opt = sum(s[2] for s in STAGES)
print(f"{'阶段':<12s} {'朴素 ms':>9s} {'占比':>7s} {'优化后':>8s} {'占比':>7s} {'省下':>7s}")
for nm, b, o, how in STAGES:
    print(f'{nm:<12s} {b:>9.2f} {b/base:>7.1%} {o:>8.2f} {o/opt:>7.1%} {b-o:>7.2f}')
print(f'{"合计":<12s} {base:>9.2f} {1.0:>7.1%} {opt:>8.2f} {1.0:>7.1%} {base-opt:>7.2f}')

infer_b = dict((s[0], s[1]) for s in STAGES)['推理 FP16']
share_b, share_o = infer_b / base, infer_b / opt
print(f'\n推理占比：{share_b:.1%}  ->  {share_o:.1%}')
print(f'端到端：{base:.2f} ms -> {opt:.2f} ms，**省 {base-opt:.2f} ms（{(base-opt)/base:.0%}），模型一个字节没改**')
assert 0.55 < share_b < 0.60 and share_o > 0.88
assert (base - opt) / base > 0.35

# 对照：如果去优化模型呢？
smaller = base - 8.20 + 6.50            # 换个小模型：8.2 -> 6.5 ms，代价约 -1.5 mAP
print(f'\n对照组：换更小的模型（8.20 -> 6.50 ms，掉约 1.5 mAP）')
print(f'  端到端 {base:.2f} -> {smaller:.2f} ms，只省 {base-smaller:.2f} ms，**而且掉精度**')
assert base - opt > 3 * (base - smaller)
print(f'✅ 优化拷贝与预处理省的 {base-opt:.2f} ms，是换小模型的 {(base-opt)/(base-smaller):.1f} 倍，且零精度代价。')
print('   新手一上来就想换 backbone；老手先画分解图、先量拷贝。')

In [ ]:
# —— 拷贝量计算器：两条最容易被忽略的收益 ——
BW_GBs = 16.0                              # 有效带宽（GB/s）

def copy_ms(nbytes, bw=BW_GBs):
    return nbytes / (bw * 1e9) * 1e3

H, W, C = 640, 640, 3
N_POS, N_CLS, N_DET = 8400, 200, 300
cases = [
    ('H2D  float32 图像',      H * W * C * 4),
    ('H2D  uint8   图像',      H * W * C * 1),
    ('D2H  raw 输出 (fp16)',   N_POS * (4 + N_CLS) * 2),
    ('D2H  最终 300 框 (fp32)', N_DET * 6 * 4),
]
print(f"{'':<24s} {'字节':>12s} {'MB':>8s} {'拷贝 ms':>9s} {'30FPS 下带宽占用':>18s}")
for nm, nb in cases:
    print(f'{nm:<24s} {nb:>12,d} {nb/1e6:>8.3f} {copy_ms(nb):>9.4f} {nb*30/1e6:>15.1f} MB/s')

h2d_ratio = cases[0][1] / cases[1][1]
d2h_ratio = cases[2][1] / cases[3][1]
print(f'\nH2D 传 uint8 而不是 float32：拷贝量降 **{h2d_ratio:.0f} 倍**')
print(f'D2H 只传最终框而不是 raw ：拷贝量降 **{d2h_ratio:.0f} 倍**')
assert abs(h2d_ratio - 4.0) < 1e-9
assert abs(d2h_ratio - 476.0) < 1e-9
saved_bw = (cases[0][1] - cases[1][1] + cases[2][1] - cases[3][1]) * 30 / 1e6
print(f'\n✅ 两条加起来每秒少占 {saved_bw:.0f} MB/s 的内存带宽 ——')
print('   车端 SoC 上带宽是所有感知任务**共享**的稀缺资源，你少占一点，别的任务就快一点。')
print('   这也是为什么「拷贝优化」的价值常常被低估：它省的不只是自己的时间。')

## 5 · Roofline：算子到底受什么限制

$I = \text{FLOPs}/\text{Bytes}$，$P = \min(P_{\text{peak}}, I\cdot BW)$，脊点 $I_{\text{ridge}} = P_{\text{peak}}/BW$。
车端 SoC 量级：FP16 峰值 42 TFLOPS、LPDDR5 带宽 204.8 GB/s。

In [ ]:
PEAK_FLOPS = 42e12          # FP16 稠密峰值 FLOP/s
BW = 204.8e9                # 内存带宽 B/s
RIDGE = PEAK_FLOPS / BW
print(f'脊点 I_ridge = {PEAK_FLOPS:.3g} / {BW:.4g} = **{RIDGE:.0f} FLOP/Byte**')
print('-> 一个算子必须「每读 1 字节做 205 次浮点运算」才配称计算受限。绝大多数算子达不到。')
assert abs(RIDGE - 205) < 3

DB = 2                      # fp16 每元素 2 字节

def conv_cost(H, W, Cin, Cout, k, groups=1):
    flops = 2.0 * H * W * Cin * Cout * k * k / groups
    byts = (H * W * Cin + H * W * Cout + Cin * (Cout // groups) * k * k) * DB
    return flops, byts

def ew_cost(n_elem, n_read, n_write, flops_per=1.0):
    return flops_per * n_elem, (n_read + n_write) * n_elem * DB

def roofline(flops, byts):
    I = flops / byts if byts else float('inf')
    t_compute = flops / PEAK_FLOPS
    t_memory = byts / BW
    return I, max(t_compute, t_memory) * 1e6, ('计算受限' if t_compute >= t_memory else '访存受限')

OPS = [
    ('3x3 conv 256->256 @40x40', *conv_cost(40, 40, 256, 256, 3)),
    ('3x3 conv 256->256 @80x80', *conv_cost(80, 80, 256, 256, 3)),
    ('1x1 conv 256->256 @40x40', *conv_cost(40, 40, 256, 256, 1)),
    ('5x5 depthwise 256 @40x40', *conv_cost(40, 40, 256, 256, 5, groups=256)),
    ('逐元素 Add @40x40x256',     *ew_cost(40 * 40 * 256, 2, 1, 1.0)),
    ('SiLU 激活 @40x40x256',      *ew_cost(40 * 40 * 256, 1, 1, 4.0)),
    ('Concat/Reformat @40x40x256', *ew_cost(40 * 40 * 256, 1, 1, 0.0)),
]
print(f"\n{'算子':<28s} {'GFLOP':>8s} {'MB':>7s} {'I':>8s} {'判定':>9s} {'耗时 μs':>9s} {'比 FLOPs 预测慢':>16s}")
res = {}
for nm, fl, by in OPS:
    I, us, kind = roofline(fl, by)
    naive_us = fl / PEAK_FLOPS * 1e6
    slow = us / naive_us if naive_us > 0 else float('inf')
    res[nm] = (I, us, kind, slow)
    st = f'{slow:>15.1f}x' if np.isfinite(slow) else f"{'∞':>16s}"
    print(f'{nm:<28s} {fl/1e9:>8.4f} {by/1e6:>7.2f} {I:>8.1f} {kind:>9s} {us:>9.1f}{st}')

assert res['3x3 conv 256->256 @40x40'][0] > RIDGE
assert res['5x5 depthwise 256 @40x40'][0] < RIDGE / 10
assert res['5x5 depthwise 256 @40x40'][3] > 15, 'depthwise 的实际耗时远超 FLOPs 预测'
print('\n⚠️  **最重要的一行是 depthwise**：')
d3 = conv_cost(40, 40, 256, 256, 3)[0]
dd = conv_cost(40, 40, 256, 256, 5, groups=256)[0]
t3 = res['3x3 conv 256->256 @40x40'][1]
td = res['5x5 depthwise 256 @40x40'][1]
print(f'    5x5 depthwise 的 FLOPs 只有 3x3 稠密卷积的 1/{d3/dd:.0f}，但延迟只快 {t3/td:.1f} 倍；')
print(f'    它的实际耗时是「按 FLOPs 推算」的 **{res["5x5 depthwise 256 @40x40"][3]:.1f} 倍**。')
assert 90 < d3 / dd < 95 and 5.0 < t3 / td < 6.5
print('✅ **FLOPs 不是延迟的好代理**，尤其在大量使用 depthwise 的轻量化架构上。')
print('   C53 模块 03 说 RTMDet 的 5x5 大核「参数量代价极小」是对的 —— 但那说的是参数量。')
print('   选型必须实测，不能查 FLOPs 表。')

In [ ]:
# —— Roofline 文本图 ——
print('可达性能 P = min(42 TFLOPS, I x 204.8 GB/s)      脊点 I = 205\n')
print(f"{'I (FLOP/B)':>12s} {'可达 P (TFLOPS)':>16s}  {'':<44s}")
for I in [0.2, 1, 4, 12.4, 40, 119, 205, 400, 670, 976, 3000]:
    P = min(PEAK_FLOPS, I * BW) / 1e12
    bar = '█' * max(1, int(44 * (P / 42.0) ** 0.5))
    tag = ''
    if abs(I - 205) < 1e-9: tag = ' ← 脊点'
    elif abs(I - 12.4) < 0.6: tag = ' ← 5x5 depthwise'
    elif abs(I - 119) < 1.5: tag = ' ← 1x1 conv'
    elif abs(I - 670) < 5: tag = ' ← 3x3 conv'
    print(f'{I:>12.1f} {P:>16.2f}  {bar}{tag}')

# —— 融合为什么收益这么大 ——
conv_f, conv_b = conv_cost(40, 40, 256, 256, 3)
_, t_conv, _ = roofline(conv_f, conv_b)
_, t_bn, _ = roofline(*ew_cost(40 * 40 * 256, 1, 1, 2.0))
_, t_act, _ = roofline(*ew_cost(40 * 40 * 256, 1, 1, 4.0))
unfused, fused = t_conv + t_bn + t_act, t_conv
print(f'\nConv -> BN -> SiLU')
print(f'  不融合: {t_conv:.1f} + {t_bn:.1f} + {t_act:.1f} = **{unfused:.1f} μs**')
print(f'  融合后: {fused:.1f} μs（BN 折进卷积权重，SiLU 融进 epilogue，中间结果不落显存）')
print(f'  **省 {1-fused/unfused:.0%}，而 FLOPs 只减少了 {(2+4)*40*40*256/conv_f:.2%}**')
assert unfused / fused > 1.2
print('✅ 融合省的从来不是计算，是「把中间结果写回显存再读回来」这一趟。')

In [ ]:
# —— 第三种瓶颈：并行度不足 / kernel 启动开销 ——
LAUNCH_US = 6.0                 # 每个 kernel 的启动开销（CPU 提交 + GPU 调度）
GPU_MS = 8.20

print(f"{'融合后层数':>10s} {'启动开销 ms':>13s} {'占 8.2ms 的':>12s} {'用 CUDA Graph 后':>17s}")
for n_layer in [60, 120, 200, 320]:
    ov = n_layer * LAUNCH_US / 1000
    graph_ov = 0.02 + n_layer * 0.15 / 1000        # graph：一次提交 + 极小的 per-node 开销
    print(f'{n_layer:>10d} {ov:>13.2f} {ov/GPU_MS:>11.1%} {graph_ov:>16.2f}')

ov200 = 200 * LAUNCH_US / 1000
assert ov200 / GPU_MS > 0.10, '200 层的启动开销就占了 8.2ms 模型的 10% 以上'
print(f'\n⚠️  200 层的启动开销 {ov200:.2f} ms = 模型的 {ov200/GPU_MS:.0%}。')
print('    典型症状：「GPU 利用率只有 40%，但换更大的模型延迟几乎不涨」。')
print('✅ 治法是 CUDA Graph：把 kernel 序列录制一次，之后每帧只提交一次。')
print('   代价：shape 必须固定（动态 shape 要每档录一个 graph），录制期间不能有 CPU 侧分支 ——')
print('   这正好和模块 04 的「稳态期不分配、不同步」纪律吻合。')

print('\n三种诊断结论与对策：')
for cond, verdict, fix in [
        ('有效算力接近峰值 (I > 205)', '计算受限', '降精度 FP16->INT8 / 剪枝蒸馏 / 通道数取 32 的倍数'),
        ('有效带宽接近峰值 (I < 205)', '访存受限', '**融合** / 降精度(也减字节) / 改 layout / 增大 batch 复用权重'),
        ('两个都远低于峰值',           '并行度不足', '增大 batch / 多流 / **CUDA Graph** / 检查 tail effect')]:
    print(f'  {cond:<26s} -> {verdict:<8s} : {fix}')

## 6 · 静默回退检查器：把「转了 TRT 但没变快」自动化查出来

输入是 engine 的逐层信息（`IEngineInspector` 的 JSON）+ 逐层耗时（`--dumpProfile`）+ 顶层计时。
五条规则跑一次不到一秒，能挡住绝大多数「优化了但没优化」的发布。

In [ ]:
def make_engine_report(n_fp16, n_fp32_light, n_fp32_conv, n_reformat,
                       subgraphs, gpu_ms, host_ms, seed=0):
    r = np.random.default_rng(seed)
    layers = []
    for i in range(n_fp16):
        layers.append(dict(name=f'conv_{i}', type='Convolution', precision='FP16',
                           ms=float(r.uniform(0.01, 0.08))))
    for i in range(n_fp32_light):
        layers.append(dict(name=f'norm_{i}', type='Normalization', precision='FP32',
                           ms=float(r.uniform(0.005, 0.02))))
    for i in range(n_fp32_conv):
        layers.append(dict(name=f'conv_fp32_{i}', type='Convolution', precision='FP32',
                           ms=float(r.uniform(0.20, 0.55))))
    for i in range(n_reformat):
        layers.append(dict(name=f'reformat_{i}', type='Reformat', precision='FP16',
                           ms=float(r.uniform(0.01, 0.05))))
    return dict(layers=layers, subgraphs=subgraphs, gpu_ms=gpu_ms, host_ms=host_ms)

RULES = [
    ('R1 子图数 == 1',                    lambda rep, s: rep['subgraphs'] == 1),
    ('R2 FP32 层耗时占比 < 10%',           lambda rep, s: s['fp32_share'] < 0.10),
    ('R3 无计算密集层落在 FP32',           lambda rep, s: s['n_heavy_fp32'] == 0),
    ('R4 Reformat 耗时占比 < 5%',          lambda rep, s: s['reformat_share'] < 0.05),
    ('R5 Host-GPU 差额 < GPU 的 25%',      lambda rep, s: s['host_gap_share'] < 0.25),
]
HEAVY = {'Convolution', 'MatMul', 'FullyConnected'}

def summarize(rep):
    tot = sum(l['ms'] for l in rep['layers'])
    fp32 = sum(l['ms'] for l in rep['layers'] if l['precision'] == 'FP32')
    rfm = sum(l['ms'] for l in rep['layers'] if l['type'] == 'Reformat')
    heavy32 = [l for l in rep['layers'] if l['precision'] == 'FP32' and l['type'] in HEAVY]
    return dict(total=tot, fp32_share=fp32 / tot, reformat_share=rfm / tot,
                n_heavy_fp32=len(heavy32), heavy_names=[l['name'] for l in heavy32][:3],
                host_gap_share=(rep['host_ms'] - rep['gpu_ms']) / rep['gpu_ms'])

def check(rep, title):
    s = summarize(rep)
    print(f'\n=== {title} ===')
    print(f"  层数 {len(rep['layers'])}  子图 {rep['subgraphs']}  "
          f"FP32 耗时占比 {s['fp32_share']:.1%}  Reformat 占比 {s['reformat_share']:.1%}")
    print(f"  GPU Compute {rep['gpu_ms']:.2f} ms   Host Latency {rep['host_ms']:.2f} ms   "
          f"差额 {rep['host_ms']-rep['gpu_ms']:.2f} ms ({s['host_gap_share']:.0%})")
    fails = []
    for name, fn in RULES:
        ok = fn(rep, s)
        print(f'  [{"PASS" if ok else "**FAIL**"}] {name}')
        if not ok:
            fails.append(name)
    if s['n_heavy_fp32']:
        print(f"     ↳ 落在 FP32 的计算密集层示例: {s['heavy_names']}")
    return fails

healthy = make_engine_report(187, 7, 0, 6, subgraphs=1, gpu_ms=8.20, host_ms=9.00, seed=1)
broken = make_engine_report(120, 71, 3, 24, subgraphs=4, gpu_ms=8.40, host_ms=14.60, seed=2)
f_ok = check(healthy, '健康的 engine')
f_bad = check(broken, '有静默回退的 engine')

assert f_ok == [], f_ok
assert len(f_bad) >= 4, f_bad
print(f'\n⚠️  注意两者的 **GPU Compute Time 差不多**（8.20 vs 8.40）——')
print('    只看 trtexec 报的 GPU 时间会得出「engine 没问题」的错误结论。')
print('    子图切分的代价不在计算里，在**同步与来回拷贝**里（Host Latency 差了 5.6 ms）。')
print('✅ 五条规则全部可自动化。Reformat 是最容易被忽略的时间小偷 ——')
print('   它不在你的 ONNX 里，是构建器为了对齐 layout/精度自己插的，算术强度为 0。')

## 7 · 批处理的延迟代价与车端多相机

$T(b) = b\,T_{\text{act}} + T_w$，每帧摊薄 $= T_{\text{act}} + T_w/b$。
收益上限完全由**权重读取占比**决定；代价是必须先等齐 $b$ 帧。

In [ ]:
T_ACT, T_W, FPS = 6.0, 2.2, 30      # 激活相关部分 / 权重读取部分（与 batch 无关）

def batch_time(b):
    return b * T_ACT + T_W

print(f"{'batch':>6s} {'总耗时':>9s} {'每帧摊薄':>10s} {'吞吐提升':>9s} "
      f"{'单相机端到端(含等待)':>22s} {'多相机(硬件同步)':>18s}")
per_prev = None
for b in [1, 2, 4, 8]:
    tot = batch_time(b); per = tot / b
    gain = batch_time(1) / per - 1
    wait = (b - 1) / FPS * 1000
    print(f'{b:>6d} {tot:>8.1f}ms {per:>9.2f}ms {gain:>8.0%} '
          f'{wait+tot:>19.1f}ms {tot:>16.1f}ms')
    if per_prev is not None:
        assert per < per_prev
    per_prev = per

assert (3 / FPS * 1000 + batch_time(4)) > 100, '单相机 batch=4 的端到端延迟超过 100 ms'
serial4, batched4 = 4 * batch_time(1), batch_time(4)
print(f'\n单相机 batch=4：每帧摊薄 8.20 -> 6.55 ms（省 1.65 ms），'
      f'代价是等 3 帧 = {3/FPS*1000:.0f} ms -> **端到端 {3/FPS*1000+batch_time(4):.1f} ms**')
print('  吞吐 +25% 是真的，延迟恶化 15 倍也是真的。任何延迟敏感系统都不做这笔交易。')
print(f'\n**多相机（4 路硬件同步曝光）**：无需等待')
print(f'  串行跑 4 路: {serial4:.1f} ms      batch 跑 4 路: {batched4:.1f} ms      '
      f'省 {1-batched4/serial4:.0%}')
assert batched4 < serial4 and (1 - batched4 / serial4) > 0.15
print('✅ 这是车端唯一真正合理的批处理场景。**前提有两条，缺一不可**：')
print('   ① 四路必须硬件同步曝光（否则「等齐」又回来了，时间戳还不一致）')
print('   ② 四路必须同模型同分辨率 —— 而 TSR 里这条经常不成立：')
print('      远处小标志需要长焦的高分辨率，侧视相机根本不跑 TSR。')

## 8 · 热降频模拟与分段 p99

一阶热模型 $T(t)=T_{\text{amb}}+PR_{\text{th}}(1-e^{-t/\tau})$，频率随温度线性下调，延迟 $\propto 1/f$。
**跑 100 帧和跑 30 分钟，是两个不同的世界。**

In [ ]:
T_AMB, P_W, R_TH, TAU = 55.0, 40.0, 1.1, 300.0     # 夏季舱内 55°C，40W，热阻 1.1°C/W
T_THR, K_THR, F_MIN = 85.0, 0.02, 0.65             # 85°C 起节流，每度降 2%，下限 0.65
BASE_MS = 8.20

def junction_temp(t_s):
    return T_AMB + P_W * R_TH * (1.0 - np.exp(-np.asarray(t_s, float) / TAU))

def freq_ratio(T):
    return np.clip(1.0 - K_THR * np.maximum(0.0, np.asarray(T, float) - T_THR), F_MIN, 1.0)

rng = np.random.default_rng(7)
DUR_MIN = 45
t_s = np.arange(DUR_MIN * 60 * FPS) / FPS
f_ratio = freq_ratio(junction_temp(t_s))
lat_th = (BASE_MS + rng.gamma(1.5, 0.18, len(t_s))) / f_ratio

BUDGET_MS = 12.0
print(f"{'时刻':>10s} {'结温 °C':>9s} {'频率比':>8s} {'p50':>8s} {'p99':>8s} "
      f"{'相对第1分钟':>12s} {'超预算率':>9s}")
seg_p99 = {}
for minute in [0, 5, 10, 20, 30, 44]:
    m = (t_s >= minute * 60) & (t_s < (minute + 1) * 60)
    seg = lat_th[m]
    p50, p99 = np.percentile(seg, [50, 99])
    seg_p99[minute] = p99
    rel = p99 / seg_p99[0] - 1 if minute else 0.0
    print(f'{minute:>7d} min {junction_temp((minute+0.5)*60):>9.1f} '
          f'{freq_ratio(junction_temp((minute+0.5)*60)):>8.2f} {p50:>8.2f} {p99:>8.2f} '
          f'{rel:>11.0%} {(seg>BUDGET_MS).mean():>9.2%}')

assert seg_p99[5] < 1.02 * seg_p99[0], '前 5 分钟还没到节流阈值，看不出问题'
assert seg_p99[30] > 1.25 * seg_p99[0], '30 分钟后 p99 应显著抬升'
print(f'\n⚠️  第 1 分钟 p99 = {seg_p99[0]:.2f} ms，第 30 分钟 p99 = {seg_p99[30]:.2f} ms，'
      f'**漂移 +{seg_p99[30]/seg_p99[0]-1:.0%}**')
print(f'    延迟预算 {BUDGET_MS} ms：开始时余量 {1-seg_p99[0]/BUDGET_MS:.0%}，'
      f'半小时后余量 {1-seg_p99[30]/BUDGET_MS:.0%}')
print('    **而这半小时恰恰是长途驾驶最需要 TSR 稳定工作的时候。**')
print(f'\n整段 p99（45 分钟一起算）= {np.percentile(lat_th, 99):.2f} ms ——')
print('    它把「前 5 分钟很快」和「后 40 分钟很慢」平均掉了，恰好掩盖了你最想看的漂移。')
print('✅ 所以验收要报**分段 p99**（1 分钟一段），看它是否收敛、收敛到哪。')
print('   还要注意：真实系统里降频会同时降功耗，P 与 f 耦合，实测曲线通常更早收敛、更平缓；')
print('   这个模型的价值不在预测精确值，而在告诉你「必须测长时间」以及给你一个外推的形状。')

In [ ]:
# —— 多任务时间片预算与降级 ——
TASKS = [('BEV 3D 检测', 12.0), ('占用栅格', 9.0), ('TSR', 8.2),
         ('车道线', 4.0), ('多目标跟踪(CPU)', 2.0)]
PERIOD_MS = 1000.0 / FPS
HEADROOM = 0.90                                  # 留 10% 余量给抖动
BUDGET = PERIOD_MS * HEADROOM

DEGRADE = [   # (任务, 手段, 省下 ms, 代价描述)
    ('车道线',   '降到 15 Hz（隔帧跑）',        2.00, '车道线更新率减半'),
    ('占用栅格', '输入分辨率 x0.8',             2.60, '远处栅格精度下降'),
    ('TSR',      'decoder 6 层 -> 3 层',        0.90, '约 -1 AP（C53 模块 04）'),
    ('全局',     '丢帧（最后手段）',            8.00, '**必须上报 frame_id**'),
]

total = sum(t[1] for t in TASKS)
print(f'周期预算 {PERIOD_MS:.1f} ms（30 FPS），留 10% 余量后可用 {BUDGET:.1f} ms')
for nm, ms in TASKS:
    print(f'  {nm:<18s} {ms:>6.1f} ms')
print(f'  {"合计":<18s} {total:>6.1f} ms   -> {"**超了**" if total > BUDGET else "OK"}')
assert total > BUDGET

applied, cur = [], total
for nm, how, save, cost in DEGRADE:
    if cur <= BUDGET:
        break
    cur -= save
    applied.append((nm, how, save, cost))
    print(f'  降级：{nm:<10s} {how:<24s} 省 {save:.2f} ms -> 合计 {cur:.2f} ms')

print(f'\n降级 {len(applied)} 项后合计 {cur:.2f} ms <= {BUDGET:.1f} ms ✅')
assert cur <= BUDGET and len(applied) == 3
assert applied[-1][0] == 'TSR' and '丢帧' not in [a[1] for a in applied]
print('⚠️  GPU kernel 默认**不可抢占** —— 一个 3 ms 的长 kernel 会把所有任务堵住 3 ms，')
print('    无论它们优先级多高。CUDA stream priority 只是调度倾向，不是硬实时保证。')
print('✅ 正确做法是时间片预算 + **提前设计好的降级路径**，而不是靠优先级。')
print('   RT-DETR 的「decoder 层数可裁」在这里体现出真实价值：它提供了一条**精度可降级**的路，')
print('   让系统优雅退化而不是直接丢帧 —— 丢帧是最后手段，且必须上报 frame_id。')

## 9 · 发布验收门禁：13 项检查的自动化实现

In [ ]:
# (编号, 维度, 指标键, 比较符, 门限, 单位)
GATE_SPEC = [
    ('01', '精度·整体',      'map_drop',          '<=', 1.0,   'mAP'),
    ('02', '精度·分桶',      'bucket_drop_max',   '<=', 2.0,   'mAP'),
    ('03', '精度·关键类',    'key_recall_drop',   '<=', 0.01,  '比例'),
    ('04', '延迟 p99',       'p99_ms',            '<=', 12.0,  'ms'),
    ('05', '延迟连续性',     'max_consec_timeout','<=', 2,     '帧'),
    ('06', '显存峰值',       'mem_ratio',         '<=', 0.80,  '占预算'),
    ('07', '长时稳定性',     'p99_drift',         '<=', 0.10,  '比例'),
    ('08', '多硬件一致性',   'hw_match_rate',     '>=', 0.999, '比例'),
    ('09', '数值一致性',     'xdiff_stages_pass', '>=', 8,     '阶段'),
    ('10', 'engine 指纹',    'fingerprint_ok',    '>=', 1,     'bool'),
    ('11', '冷启动',         'cold_start_s',      '<=', 6.0,   's'),
    ('12', '回退方案',       'fault_cases_pass',  '>=', 3,     '用例'),
    ('13', '可观测性',       'telemetry_fields',  '>=', 6,     '字段'),
]

def release_gate(metrics, spec=GATE_SPEC):
    rows, failures = [], []
    for num, dim, key, op, thr, unit in spec:
        v = metrics[key]
        ok = (v <= thr) if op == '<=' else (v >= thr)
        margin = (1 - v / thr) if op == '<=' and thr else ((v / thr - 1) if thr else 0.0)
        rows.append((num, dim, v, op, thr, unit, ok, margin))
        if not ok:
            failures.append((num, dim, v, op, thr))
    return rows, failures

def print_gate(title, metrics):
    rows, fails = release_gate(metrics)
    print(f'\n═══ {title} ═══')
    print(f"{'#':>3s} {'维度':<14s} {'实测':>10s} {'门限':>12s} {'余量':>8s} {'判定':>8s}")
    for num, dim, v, op, thr, unit, ok, margin in rows:
        print(f'{num:>3s} {dim:<14s} {v:>10.4g} {op+" "+format(thr, ".4g"):>12s} '
              f'{margin:>7.0%} {"PASS" if ok else "**FAIL**":>8s}')
    print(f'  -> {"✅ 全部通过，可发布" if not fails else "❌ %d 项未通过，禁止发布" % len(fails)}')
    return fails

GOOD = dict(map_drop=0.62, bucket_drop_max=1.35, key_recall_drop=0.004, p99_ms=10.4,
            max_consec_timeout=2, mem_ratio=0.71, p99_drift=0.07, hw_match_rate=0.9995,
            xdiff_stages_pass=8, fingerprint_ok=1, cold_start_s=4.3,
            fault_cases_pass=3, telemetry_fields=9)
BAD = dict(GOOD, bucket_drop_max=4.80, max_consec_timeout=37, p99_drift=0.31)

assert print_gate('候选 A（v2.4.1）', GOOD) == []
fails_b = print_gate('候选 B（v2.5.0-rc1）', BAD)
assert len(fails_b) == 3
assert {f[0] for f in fails_b} == {'02', '05', '07'}
print('\n候选 B 的三项失败，恰好指向三条不同的排查路径：')
print('  02 分桶掉点 -> **先查坐标变换（模块 04），再怀疑量化**（4px 偏移在 12px 框上是致命的）')
print('  05 最长连续超时 37 帧 -> 成簇超时，查热节流与多任务抢占（第 7、8 节）')
print('  07 分段 p99 漂移 31% -> 长时间稳定性不达标，查散热与稳态功耗')

In [ ]:
# —— 余量趋势：门禁只能挡住已经越界的，趋势才能提前告诉你要撞墙 ——
HISTORY = [('v2.1.0', 9.1), ('v2.2.0', 9.4), ('v2.3.0', 9.6),
           ('v2.4.0', 10.0), ('v2.4.1', 10.4)]
THR = 12.0
print(f"{'版本':<10s} {'p99 ms':>8s} {'余量':>8s} {'判定':>8s} {'环比':>8s}")
prev = None
for v, p in HISTORY:
    d = f'{p-prev:+.2f}' if prev is not None else '—'
    print(f'{v:<10s} {p:>8.2f} {1-p/THR:>7.0%} {"PASS":>8s} {d:>8s}')
    prev = p

xs = np.arange(len(HISTORY), dtype=float)
ys = np.array([p for _, p in HISTORY])
slope, intercept = np.polyfit(xs, ys, 1)
n_left = (THR - intercept) / slope - (len(HISTORY) - 1)
print(f'\n线性外推：每个版本 p99 涨 {slope:.3f} ms，'
      f'再过 **{n_left:.1f} 个版本**就会撞上 {THR} ms 的门限')
assert slope > 0 and 0 < n_left < 8
print('⚠️  这五个版本**每一次都「通过」**，但趋势明摆着说下一次就要撞墙。')
print('✅ 所以发布报告必须输出结构化的「实测值 / 门限 / 余量 / 环比」，而不只是 pass/fail。')
print('   而这一切的前提是第 13 项（可观测性）—— **没有打点的系统，你连趋势都画不出来。**')

## ✏️ 练习 1：延迟报告生成器

实现 `latency_report(lat, budget, fps)`，返回一个 dict，键为：

- `p50` / `p99` / `p999` / `max`：用 `np.percentile`（线性插值，默认行为）
- `eps`：超过 `budget` 的比例（严格大于）
- `per_hour`：`eps * fps * 3600`
- `max_consec`：**最长连续超时段**（连续多少帧都超过 budget）

In [ ]:
def latency_report(lat, budget, fps):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测（数字可口算）——
x = np.arange(1.0, 101.0)                      # 1..100，超过 95 的正好是 96..100 共 5 个
rep = latency_report(x, budget=95.0, fps=10)
print({k: round(v, 4) for k, v in rep.items()})
assert abs(rep['p50'] - 50.5) < 1e-9
assert abs(rep['p99'] - 99.01) < 1e-9
assert abs(rep['max'] - 100.0) < 1e-9
assert abs(rep['eps'] - 0.05) < 1e-12
assert abs(rep['per_hour'] - 1800.0) < 1e-9
assert rep['max_consec'] == 5, '96..100 连在一起'

# 同一组数值重排：所有分位数不变，最长连续超时段变了
y = np.zeros(100)
big = [0, 20, 40, 60, 80]
y[big] = x[95:]
y[[i for i in range(100) if i not in big]] = x[:95]
rep2 = latency_report(y, budget=95.0, fps=10)
assert abs(rep2['p99'] - rep['p99']) < 1e-9 and abs(rep2['eps'] - rep['eps']) < 1e-12
assert rep2['max_consec'] == 1
print(f"\n重排前 max_consec={rep['max_consec']}，重排后 max_consec={rep2['max_consec']}，"
      f"而 p50/p99/eps 完全相同")
print('✅ 练习 1 通过：p99 和「最长连续超时段」是两个正交的指标，必须都报。')

## ✏️ 练习 2：Roofline 判定器

实现 `roofline_us(flops, byts, peak=PEAK_FLOPS, bw=BW)`，返回 `(I, kind, us)`：

- `I = flops / byts`（`byts` 为 0 时返回 `inf`）
- `t_compute = flops/peak`，`t_memory = byts/bw`，耗时取**两者的较大值**
- `kind`：`t_compute >= t_memory` 时是 `'计算受限'`，否则 `'访存受限'`
- `us` 单位是微秒

In [ ]:
def roofline_us(flops, byts, peak=PEAK_FLOPS, bw=BW):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
f3, b3 = conv_cost(40, 40, 256, 256, 3)
I3, k3, u3 = roofline_us(f3, b3)
assert k3 == '计算受限' and abs(u3 - 44.94) < 0.05, (I3, k3, u3)
assert abs(I3 - 669.8) < 0.5

fd, bd = conv_cost(40, 40, 256, 256, 5, groups=256)
Id, kd, ud = roofline_us(fd, bd)
assert kd == '访存受限' and abs(ud - 8.06) < 0.05, (Id, kd, ud)
assert Id < RIDGE / 10

fa, ba = ew_cost(40 * 40 * 256, 2, 1, 1.0)      # 逐元素 Add
Ia, ka, ua = roofline_us(fa, ba)
assert ka == '访存受限' and Ia < 1.0

fc, bc = ew_cost(40 * 40 * 256, 1, 1, 0.0)      # 纯搬运（concat/reformat）
Ic, kc, uc = roofline_us(fc, bc)
assert Ic == 0.0 and kc == '访存受限'

print(f"{'算子':<22s} {'I':>9s} {'判定':>9s} {'μs':>8s} {'FLOPs 预测 μs':>14s}")
for nm, fl, by in [('3x3 conv', f3, b3), ('5x5 depthwise', fd, bd),
                   ('逐元素 Add', fa, ba), ('Concat/Reformat', fc, bc)]:
    I, k, u = roofline_us(fl, by)
    print(f'{nm:<22s} {I:>9.1f} {k:>9s} {u:>8.2f} {fl/PEAK_FLOPS*1e6:>14.3f}')
print(f'\n✅ 练习 2 通过：depthwise 的实际耗时是 FLOPs 预测的 {ud/(fd/PEAK_FLOPS*1e6):.1f} 倍。')
print('   任何以 FLOPs 为唯一目标的架构搜索，都会在这类算子上系统性地骗自己。')

## ✏️ 练习 3：分段 p99 与漂移

实现两个函数：

- `segment_p99(lat, t_s, seg_seconds)`：把时间轴按 `seg_seconds` 切段，
  返回每段的 p99 组成的数组（时间戳 `t_s` 与 `lat` 一一对应；空段跳过）
- `drift(segs)`：返回 `segs.max() / segs[0] - 1`（相对第一段的最大漂移）

In [ ]:
def segment_p99(lat, t_s, seg_seconds):
    # TODO
    raise NotImplementedError

def drift(segs):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测（构造一个阶跃：前 5 分钟 8ms，后 5 分钟 12ms）——
tt = np.arange(0, 600, 1.0 / 30)
ll = np.where(tt < 300, 8.0, 12.0)
segs = segment_p99(ll, tt, 60)
print('分段 p99:', np.round(segs, 3))
assert len(segs) == 10, len(segs)
assert np.allclose(segs[:5], 8.0) and np.allclose(segs[5:], 12.0)
assert abs(drift(segs) - 0.5) < 1e-9

# 接到第 8 节的热降频序列上
segs_th = segment_p99(lat_th, t_s, 60)
print(f'\n热降频序列：{len(segs_th)} 段，第 1 段 p99 = {segs_th[0]:.2f} ms，'
      f'最后一段 = {segs_th[-1]:.2f} ms，漂移 = {drift(segs_th):.0%}')
assert len(segs_th) == 45
assert drift(segs_th) > 0.25, '30 分钟以上的漂移应超过 25%'
whole = float(np.percentile(lat_th, 99))
print(f'整段 p99 = {whole:.2f} ms —— 它介于两端之间，把漂移平均掉了。')
assert segs_th[0] < whole < segs_th[-1] + 0.1
print('✅ 练习 3 通过：**整段 p99 会掩盖漂移，分段 p99 才看得见它。**')
print('   验收门槛写「分段 p99 漂移 <= 10%」，而不是「整段 p99 <= X」。')

## ✏️ 练习 4：时间片降级求解器

实现 `degrade_to_fit(total_ms, degrade_list, budget_ms)`：

- `degrade_list` 是 `[(任务名, 省下 ms), ...]`，**按优先级从低到高排好序**（先牺牲优先级低的）
- 按顺序应用降级，直到 `total <= budget` 或降级手段用完
- 返回 `(applied_names, final_ms)`；一开始就满足预算时返回 `([], total_ms)`

In [ ]:
def degrade_to_fit(total_ms, degrade_list, budget_ms):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
DEG = [('车道线', 2.0), ('占用栅格', 2.6), ('TSR', 0.9), ('丢帧', 8.0)]

a1, f1 = degrade_to_fit(35.2, DEG, 30.0)
assert a1 == ['车道线', '占用栅格', 'TSR'] and abs(f1 - 29.7) < 1e-9, (a1, f1)

a2, f2 = degrade_to_fit(35.2, DEG, 34.0)
assert a2 == ['车道线'] and abs(f2 - 33.2) < 1e-9, (a2, f2)

a3, f3 = degrade_to_fit(29.0, DEG, 30.0)
assert a3 == [] and abs(f3 - 29.0) < 1e-9, '本来就够，不该降级'

a4, f4 = degrade_to_fit(60.0, DEG, 30.0)
assert a4 == ['车道线', '占用栅格', 'TSR', '丢帧'] and f4 > 30.0, '手段用完仍不够，要报警'

for tot, bud in [(35.2, 30.0), (35.2, 34.0), (29.0, 30.0), (60.0, 30.0)]:
    ap, fin = degrade_to_fit(tot, DEG, bud)
    status = 'OK' if fin <= bud else '**仍然超预算 -> 必须报警并回退模型**'
    print(f'负载 {tot:>5.1f} ms / 预算 {bud:>5.1f} ms -> 降级 {str(ap):<44s} 剩 {fin:>5.2f} ms  {status}')
print('\n✅ 练习 4 通过：降级路径必须**提前设计好并测试过** —— 它本身就是要上车的代码。')
print('   注意最后一行：手段用完仍不够时，系统必须报警而不是默默丢帧。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def latency_report(lat, budget, fps):
    lat = np.asarray(lat, dtype=float)
    over = lat > budget
    best = cur = 0
    for v in over:
        cur = cur + 1 if v else 0
        if cur > best:
            best = cur
    eps = float(over.mean())
    p50, p99, p999 = np.percentile(lat, [50, 99, 99.9])
    return dict(p50=float(p50), p99=float(p99), p999=float(p999), max=float(lat.max()),
                eps=eps, per_hour=eps * fps * 3600, max_consec=int(best))

In [ ]:
# 练习 2 参考答案
def roofline_us(flops, byts, peak=PEAK_FLOPS, bw=BW):
    I = (flops / byts) if byts else float('inf')
    t_compute = flops / peak
    t_memory = byts / bw
    kind = '计算受限' if t_compute >= t_memory else '访存受限'
    return I, kind, max(t_compute, t_memory) * 1e6

In [ ]:
# 练习 3 参考答案
def segment_p99(lat, t_s, seg_seconds):
    lat = np.asarray(lat, dtype=float); t_s = np.asarray(t_s, dtype=float)
    idx = (t_s // seg_seconds).astype(int)
    out = []
    for k in range(idx.min(), idx.max() + 1):
        m = idx == k
        if m.any():
            out.append(float(np.percentile(lat[m], 99)))
    return np.array(out)

def drift(segs):
    segs = np.asarray(segs, dtype=float)
    return float(segs.max() / segs[0] - 1.0)

In [ ]:
# 练习 4 参考答案
def degrade_to_fit(total_ms, degrade_list, budget_ms):
    applied, cur = [], float(total_ms)
    for name, save in degrade_list:
        if cur <= budget_ms:
            break
        cur -= save
        applied.append(name)
    return applied, cur

---
## 🧪 真实工程胶囊：benchmark 脚本骨架 + 发布验收流程

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 第一部分：一次「说得清楚」的 benchmark 必须记录什么
#   把它写成脚本头部的元数据，跟结果一起存档 —— 缺任何一项，这个数字都不可比
# ══════════════════════════════════════════════════════════════════════
bench_meta:
  hardware:      Orin AGX 64GB
  power_mode:    MAXN            # ← **必须声明**，不同档延迟可差 1.5-2 倍
  clocks:        "jetson_clocks --show 的输出"
  trt_version:   10.x
  driver:        "…"
  precision:     FP16            # 或 INT8 + 校准集版本
  batch:         1
  input_shape:   [1,3,640,640]
  scope:         end_to_end      # end_to_end | gpu_only  ← 口径
  includes:      [preprocess, h2d, infer, postprocess, d2h]
  warmup:        500             # 次
  frames:        10000           # ← p99 的最低采样量；估 p99.9 要 10 万
  frame_source:  "real_sequence_urban_v3"   # **不是同一张图跑 N 次**
  duration_min:  45              # ← 覆盖热稳态
  ambient_c:     55              # ← 目标环境温度，不是空调房

# ══════════════════════════════════════════════════════════════════════
# 第二部分：trtexec 常用姿势
# ══════════════════════════════════════════════════════════════════════
# ① 稳态延迟（分开性能跑与 profile 跑，避免观察者效应）
#   trtexec --loadEngine=tsr.plan --shapes=images:1x3x640x640 #           --warmUp=2000 --duration=120 --avgRuns=100 --percentile=99
# ② 逐层耗时（**单独一次跑**）
#   trtexec --loadEngine=tsr.plan --dumpProfile --separateProfileRun #           --exportProfile=layers.json
# ③ 逐层精度与 tactic（查静默回退）
#   trtexec --loadEngine=tsr.plan --dumpLayerInfo --exportLayerInfo=layers_info.json
# ④ CUDA Graph（小模型上常有 10-25% 收益）
#   trtexec --loadEngine=tsr.plan --useCudaGraph
# ⑤ 动态 shape 三档都要测
#   for s in 1x3x480x480 1x3x640x640 1x3x800x800; do trtexec --shapes=images:$s ...; done
#
# 读输出时分清四个数：GPU Compute Time / Enqueue Time / Host Latency / Throughput
#   · Enqueue ≈ GPU Compute      -> CPU 提交跟不上，上 CUDA Graph
#   · Host Latency >> GPU Compute -> 拷贝或子图切换的开销（查静默回退）

# ══════════════════════════════════════════════════════════════════════
# 第三部分：静默回退五条规则（解析 layers_info.json + layers.json）
# ══════════════════════════════════════════════════════════════════════
#   R1 子图数 == 1                              （ORT 用 TRT EP 时尤其要看）
#   R2 FP32 层耗时占比 < 10%
#   R3 无 Convolution/MatMul 落在 FP32
#   R4 Reformat 层耗时占比 < 5%                 ← 它不在你的 ONNX 里，是构建器插的
#   R5 (Host Latency - GPU Compute) / GPU Compute < 25%

# ══════════════════════════════════════════════════════════════════════
# 第四部分：发布验收 13 项（每一项对应一个真实事故类型）
# ══════════════════════════════════════════════════════════════════════
#   01 整体 mAP 掉幅 <= 1.0
#   02 **每一个像素尺寸桶**的 AP 掉幅 <= 2.0     ← 平均会掩盖小目标崩塌
#   03 安全关键类召回 >= baseline - 1%
#   04 端到端 p99 <= 预算；p99.9 <= 预算 x 1.1   （>= 10000 帧真实序列）
#   05 **最长连续超时段 <= 2 帧**                 ← 最常被漏掉
#   06 显存峰值 <= 分配预算的 80%
#   07 **>= 30 min 连续跑，分段 p99 漂移 <= 10%，RSS 斜率 ≈ 0**   ← 进夜间 CI
#   08 多硬件/多驱动一致性：框匹配率 >= 99.9%
#   09 与 Python 参考实现的**逐阶段对拍 8 个阶段全过**（模块 04）
#   10 engine 指纹校验（GPU 型号+驱动+TRT 版本+ONNX 哈希），不匹配拒绝加载
#   11 冷启动（engine 反序列化 + warmup）<= 整车上电预算
#   12 **回退方案本身被测试过**：注入「加载失败/推理超时/输出异常」三种故障 ← 做成自动用例
#   13 可观测性：分阶段耗时、候选数、检出数、丢帧数全部打点
#
# 报告格式：每项输出 [实测值 / 门限 / 余量% / 环比]，不只是 pass/fail
#   余量趋势比单次判定更有价值 —— 连续几个版本 p99 稳定上涨，说明下一版就要撞墙

# ══════════════════════════════════════════════════════════════════════
# 第五部分：车端特有的坑（数据中心经验会骗你）
# ══════════════════════════════════════════════════════════════════════
#   · 批处理只在**多相机硬件同步**时合理；单相机 batch=4 要白等 100 ms
#   · 动态 batch 的三笔额外成本：构建时间、非 opt 档的 tactic 损耗、按 max 预留显存
#   · CUDA stream priority 不是硬实时保证；GPU kernel 默认不可抢占
#   · 用**时间片预算 + 提前设计好的降级路径**排期，而不是靠优先级
#   · 关键线程绑核 + SCHED_FIFO + isolcpus，压住 CPU 侧的调度抖动
#   · MPS / MIG 在车端 SoC 上通常不可用
#   · **任何延迟数字都必须声明功耗模式**；开发板 MAXN 测出的数不代表量产域控
'''
print(RECIPE)
for token in ['power_mode', 'separateProfileRun', 'useCudaGraph', 'Reformat',
              '最长连续超时段', '分段 p99', '逐阶段对拍', '硬件同步', 'SCHED_FIFO']:
    assert token in RECIPE, token
print('✅ 胶囊覆盖：benchmark 元数据 / trtexec 姿势 / 静默回退五规则 / 验收 13 项 / 车端专有坑')

### 小结

- **测延迟有七宗罪，最贵的两条是「没 warmup」和「没同步」。** 不丢 warmup 会高估 26%；
  不同步测出的是 CPU 提交时间，会得到 22000 FPS 这种荒谬值。
  **报 FPS 必须声明口径**：GPU Compute / Enqueue / Host Latency / Throughput 是四个不同的数。
- **p99 需要样本量。** $\sigma_{\text{rank}}=\sqrt{np(1-p)}$：$n=100$ 时只有 1 个样本超过 p99，
  你报的「p99」就是最大值；**10 000 帧是下限**（30 FPS 下 5.5 分钟），估 p99.9 要 10 万帧。
- **「p99 达标」= 每 3.3 秒丢一帧。** 而且必须同时报**最长连续超时段**——
  同一组延迟值重排一下顺序，所有分位数完全相同，最长连续超时段却能从 2 帧变成 2000 帧
  （连续 67 秒失明）。独立抖动治调度，成簇超时治热与抢占。
- **推理往往只占一半。** 分解后：预处理 3.20 → 0.35、后处理 2.10 → 0.35、
  H2D 传 uint8 降 4 倍、D2H 只传框降 476 倍 —— **端到端省 37%，模型一个字节没改**，
  是换小模型（省 12% 且掉 1.5 mAP）的 3 倍收益。**新手换 backbone，老手先量拷贝。**
- **Roofline 的脊点是 205 FLOP/Byte，绝大多数算子够不着。**
  **5×5 depthwise 的实际耗时是 FLOPs 预测的 16.5 倍**——FLOPs 不是延迟的好代理。
  融合省的不是计算而是「中间结果的一趟往返访存」（Conv+BN+SiLU 省 26%，FLOPs 只减 0.13%）。
  第三种瓶颈是并行度不足：200 层的 kernel 启动开销就占 8.2 ms 模型的 15%，用 CUDA Graph 治。
- **静默回退可以五条规则自动查**：子图数、FP32 耗时占比、计算密集层的精度、Reformat 占比、
  Host−GPU 差额。**注意健康与回退两个 engine 的 GPU Compute Time 几乎一样**——
  只看 GPU 时间会得出「engine 没问题」的错误结论。
- **车端三重约束**：功耗模式（不声明就是无效数字）、热降频（第 1 分钟 p99 9.2 ms，
  第 30 分钟 12.7 ms，**漂移 38%**，整段 p99 会把它平均掉 → 必须报分段 p99）、
  多任务抢占（GPU kernel 不可抢占 → 靠时间片预算 + 提前设计好的降级路径，不是靠优先级）。
- **13 项验收门禁，最容易被跳过的是第 7 项（长时稳定性，因为慢）和第 12 项（回退方案测试，
  因为"应该不会发生"）——而它们正是事故的两大来源。** 报告要输出余量与环比，
  因为门禁只能挡住已经越界的，**趋势才能提前告诉你要撞墙**。

至此 C60 全课结束：一致性（m01–m04）+ 性能（m05）= 一个能上车、能验收、能追溯的推理系统。
下一站建议：**C61 · 检测工程实战与面试实务** —— 把这九门课组织成能讲十分钟的深度故事。